In [6]:
# fashion_qcnn_4class_pairwise_fragment_localmeans_topbottom_hur8_prob_ce.py

import random
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


# ============================================================
# Config
# ============================================================

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_QUBITS = 8
IMG_SIZE = 16
N_CLASSES = 4

# Fashion-MNIST:
# 0 = T-shirt/top
# 1 = Trouser
# 7 = Sneaker
# 8 = Bag
FASHION_CLASSES = [0, 1, 7, 8]
CLASS_MAP = {0: 0, 1: 1, 7: 2, 8: 3}

TRAIN_PER_CLASS = 2000
VAL_PER_CLASS = 400
TEST_PER_CLASS = 200

BATCH_SIZE = 24
EPOCHS = 15
LR = 0.001

PERIODIC_BOUNDARY = False

# Pairwise fragment encoding with local 2x2 mean features:
# 16 x 16 image -> four 8 x 8 patches -> four qubit pairs.
# Each 8 x 8 patch is compressed into sixteen 2 x 2 local means.
PATCH_SIZE = 8
SUBPATCH_SIZE = 2

N_PATCHES = 4
N_PATCH_GROUPS = 2       # group 0 = top patches, group 1 = bottom patches

FEATURES_PER_PATCH = 16
FEATURES_PER_ENCODING_STEP = 4
N_ENCODING_STEPS = FEATURES_PER_PATCH // FEATURES_PER_ENCODING_STEP  # 4


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# Dataset utilities
# ============================================================

class RemapFashionMNIST(torch.utils.data.Dataset):
    def __init__(self, base_dataset, class_map):
        self.base_dataset = base_dataset
        self.class_map = class_map

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x, y = self.base_dataset[idx]
        return x, self.class_map[int(y)]


def make_balanced_indices(dataset, class_map, n_per_class, offset_per_class=0):
    buckets = {c: [] for c in class_map.keys()}

    for idx in range(len(dataset)):
        _, y = dataset[idx]
        y = int(y)

        if y in buckets:
            buckets[y].append(idx)

    selected = []
    rng = np.random.default_rng(SEED)

    for c in class_map.keys():
        indices = np.array(buckets[c])
        rng.shuffle(indices)

        start = offset_per_class
        end = offset_per_class + n_per_class

        if end > len(indices):
            raise ValueError(
                f"Not enough samples for class {c}. "
                f"Requested indices up to {end}, available {len(indices)}."
            )

        selected.extend(indices[start:end].tolist())

    rng.shuffle(selected)
    return selected


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


def load_data():
    train_full = datasets.FashionMNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform,
    )

    test_full = datasets.FashionMNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform,
    )

    train_idx = make_balanced_indices(
        train_full,
        CLASS_MAP,
        TRAIN_PER_CLASS,
        offset_per_class=0,
    )

    val_idx = make_balanced_indices(
        train_full,
        CLASS_MAP,
        VAL_PER_CLASS,
        offset_per_class=TRAIN_PER_CLASS,
    )

    test_idx = make_balanced_indices(
        test_full,
        CLASS_MAP,
        TEST_PER_CLASS,
        offset_per_class=0,
    )

    train_ds = RemapFashionMNIST(Subset(train_full, train_idx), CLASS_MAP)
    val_ds = RemapFashionMNIST(Subset(train_full, val_idx), CLASS_MAP)
    test_ds = RemapFashionMNIST(Subset(test_full, test_idx), CLASS_MAP)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    return train_loader, val_loader, test_loader


# ============================================================
# Patch extraction with local 2x2 means
# ============================================================

def patch_8x8_to_2x2_means(patch):
    """
    Convert one 8x8 patch into sixteen local 2x2 means.

    Input:
        patch shape: (B, 8, 8)

    Output:
        features shape: (B, 16)
    """
    B = patch.shape[0]

    # (B, 8, 8) -> (B, 4, 2, 4, 2)
    patch = patch.reshape(B, 4, 2, 4, 2)

    # Mean over each internal 2x2 sub-patch
    # output shape: (B, 4, 4)
    means = patch.mean(dim=(2, 4))

    # Flatten 4x4 -> 16
    return means.reshape(B, -1)


def images_to_four_patches(x):
    """
    Convert a batch of images into four patch-feature vectors.

    Input:
        x shape: (B, 1, 16, 16)

    Output:
        patches shape: (B, 4, 16)

    Patch order:
        patch 0 = top-left
        patch 1 = top-right
        patch 2 = bottom-left
        patch 3 = bottom-right
    """
    img = x[:, 0, :, :]  # (B, 16, 16)

    p0 = img[:, 0:8, 0:8]
    p1 = img[:, 0:8, 8:16]
    p2 = img[:, 8:16, 0:8]
    p3 = img[:, 8:16, 8:16]

    f0 = patch_8x8_to_2x2_means(p0)
    f1 = patch_8x8_to_2x2_means(p1)
    f2 = patch_8x8_to_2x2_means(p2)
    f3 = patch_8x8_to_2x2_means(p3)

    patches = torch.stack([f0, f1, f2, f3], dim=1)

    # Feature values are in [0,1]. Convert them to angles in [0, pi].
    patches = np.pi * patches

    return patches


# ============================================================
# Pairwise fragment encoding
# ============================================================

def encode_patch_on_qubit_pair(patch, theta_enc_group, wires):
    """
    Encode one 8x8 image patch, already compressed into 16 local
    2x2 mean features, into one pair of qubits.

    patch:
        shape (B, 16)

    theta_enc_group:
        shape (4, 4)

    wires:
        pair of qubits, e.g. (0, 1)
    """
    a, b = wires

    for s in range(N_ENCODING_STEPS):
        base = 4 * s

        x0 = patch[:, base + 0]
        x1 = patch[:, base + 1]
        x2 = patch[:, base + 2]
        x3 = patch[:, base + 3]

        # Data-dependent encoding gates
        qml.RY(x0, wires=a)
        qml.RZ(x1, wires=a)
        qml.RY(x2, wires=b)
        qml.RZ(x3, wires=b)

        # Trainable local processing
        qml.CNOT(wires=[a, b])
        qml.RY(theta_enc_group[s, 0], wires=a)
        qml.RY(theta_enc_group[s, 1], wires=b)

        qml.CNOT(wires=[b, a])
        qml.RZ(theta_enc_group[s, 2], wires=a)
        qml.RZ(theta_enc_group[s, 3], wires=b)


def pairwise_fragment_encoding(patches, theta_enc):
    """
    Encode four image patches into four qubit pairs.

    patches shape:
        (B, 4, 16)

    theta_enc shape:
        (2, 4, 4)

    Parameter sharing:
        patch 0 = top-left     -> theta_enc[0]
        patch 1 = top-right    -> theta_enc[0]
        patch 2 = bottom-left  -> theta_enc[1]
        patch 3 = bottom-right -> theta_enc[1]
    """
    pair_wires = [
        (0, 1),
        (2, 3),
        (4, 5),
        (6, 7),
    ]

    for patch_idx, wires in enumerate(pair_wires):
        if patch_idx in [0, 1]:
            group_idx = 0  # top patches
        else:
            group_idx = 1  # bottom patches

        encode_patch_on_qubit_pair(
            patches[:, patch_idx, :],
            theta_enc[group_idx],
            wires,
        )


# ============================================================
# Quantum circuit blocks: Hur circuit 8 + Hur pooling
# ============================================================

def hur_convolution_circuit8(theta, wires):
    """
    Hur et al. convolutional circuit 8.

    10 trainable parameters.
    """
    a, b = wires

    qml.RX(theta[0], wires=a)
    qml.RX(theta[1], wires=b)

    qml.RZ(theta[2], wires=a)
    qml.RZ(theta[3], wires=b)

    qml.RX(theta[4], wires=a)
    qml.RX(theta[5], wires=b)

    qml.CNOT(wires=[a, b])

    qml.RX(theta[6], wires=a)
    qml.RX(theta[7], wires=b)

    qml.RZ(theta[8], wires=a)
    qml.RZ(theta[9], wires=b)


def convolution_layer_on_wires(theta_conv, active_wires):
    """
    QCNN-style convolution layer on the currently active wires.
    """
    even_pairs = []
    shifted_pairs = []

    for i in range(0, len(active_wires) - 1, 2):
        even_pairs.append((active_wires[i], active_wires[i + 1]))

    for i in range(1, len(active_wires) - 1, 2):
        shifted_pairs.append((active_wires[i], active_wires[i + 1]))

    if PERIODIC_BOUNDARY and len(active_wires) > 2:
        shifted_pairs.append((active_wires[-1], active_wires[0]))

    for pair in even_pairs:
        hur_convolution_circuit8(theta_conv, pair)

    for pair in shifted_pairs:
        hur_convolution_circuit8(theta_conv, pair)


def controlled_rx_on_zero(angle, control, target):
    """
    Controlled-RX activated when the control qubit is |0>.
    """
    qml.PauliX(wires=control)
    qml.CRX(angle, wires=[control, target])
    qml.PauliX(wires=control)


def hur_pooling_pair(theta_pool, control, target):
    """
    Hur et al. pooling block:
      - CRZ(theta_0), control |1>
      - CRX(theta_1), control |0>
    """
    qml.CRZ(theta_pool[0], wires=[control, target])
    controlled_rx_on_zero(theta_pool[1], control, target)


def hur_pooling_layer(theta_pool, pool_pairs):
    """
    Generic Hur pooling layer.
    """
    for control, target in pool_pairs:
        hur_pooling_pair(theta_pool, control, target)


# ============================================================
# QCNN model
# ============================================================

class FashionQCNN4Class(nn.Module):
    def __init__(self):
        super().__init__()

        self.dev = qml.device("default.qubit", wires=N_QUBITS)

        # Top-bottom shared encoding parameters.
        # shape:
        #   2 groups:
        #       group 0 = top-left/top-right
        #       group 1 = bottom-left/bottom-right
        #   4 encoding steps
        #   4 angles per step
        self.theta_enc = nn.Parameter(
            0.01 * torch.randn(N_PATCH_GROUPS, N_ENCODING_STEPS, 4)
        )

        # Layer 1: convolution on 8 qubits + pooling 8 -> 4
        self.theta_conv1 = nn.Parameter(0.01 * torch.randn(10))
        self.theta_pool1 = nn.Parameter(0.01 * torch.randn(2))

        # Layer 2: convolution on 4 retained qubits + pooling 4 -> 2
        self.theta_conv2 = nn.Parameter(0.01 * torch.randn(10))
        self.theta_pool2 = nn.Parameter(0.01 * torch.randn(2))

        self.wires_8 = [0, 1, 2, 3, 4, 5, 6, 7]
        self.wires_4 = [0, 2, 4, 6]

        self.output_wires = [0, 4]

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(
            patches,
            theta_enc,
            theta_conv1,
            theta_pool1,
            theta_conv2,
            theta_pool2,
        ):
            # Pairwise fragment encoding with top/bottom parameter sharing
            pairwise_fragment_encoding(patches, theta_enc)

            # Layer 1: 8 -> 4
            convolution_layer_on_wires(theta_conv1, self.wires_8)

            hur_pooling_layer(
                theta_pool1,
                pool_pairs=[
                    (1, 0),
                    (3, 2),
                    (5, 4),
                    (7, 6),
                ],
            )

            # Layer 2: 4 -> 2
            convolution_layer_on_wires(theta_conv2, self.wires_4)

            hur_pooling_layer(
                theta_pool2,
                pool_pairs=[
                    (2, 0),
                    (6, 4),
                ],
            )

            return qml.probs(wires=self.output_wires)

        self.qnode = qnode

    def forward(self, x):
        """
        x shape:
            (B, 1, 16, 16)

        returns:
            probs shape (B, 4)
        """
        patches = images_to_four_patches(x)

        probs = self.qnode(
            patches,
            self.theta_enc,
            self.theta_conv1,
            self.theta_pool1,
            self.theta_conv2,
            self.theta_pool2,
        )

        return probs.float()


# ============================================================
# Probability cross-entropy loss
# ============================================================

def probability_cross_entropy_loss(probs, labels, eps=1e-8):
    """
    L = - mean log P(y)
    """
    probs = torch.clamp(probs, min=eps, max=1.0)

    true_probs = probs[
        torch.arange(probs.shape[0], device=probs.device),
        labels,
    ]

    loss = -torch.log(true_probs).mean()
    return loss


# ============================================================
# Metrics
# ============================================================

def accuracy(probs, labels):
    preds = torch.argmax(probs, dim=1)
    return (preds == labels).float().mean().item()


def grad_norm(model):
    total = 0.0

    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.detach().pow(2).sum().item()

    return total ** 0.5


def confusion_matrix(model, loader):
    model.eval()

    cm = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.int64)

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            probs = model(x)
            preds = torch.argmax(probs, dim=1)

            for true_label, pred_label in zip(y.cpu(), preds.cpu()):
                cm[true_label, pred_label] += 1

    return cm


# ============================================================
# Training / evaluation
# ============================================================

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_grad_norm = 0.0
    total_n = 0
    n_batches = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        probs = model(x)
        loss = probability_cross_entropy_loss(probs, y)

        if is_train:
            loss.backward()

            batch_grad_norm = grad_norm(model)
            total_grad_norm += batch_grad_norm
            n_batches += 1

            optimizer.step()

        batch_size = x.shape[0]

        total_loss += loss.item() * batch_size
        total_acc += accuracy(probs.detach(), y) * batch_size
        total_n += batch_size

    avg_loss = total_loss / total_n
    avg_acc = total_acc / total_n

    if is_train:
        avg_grad_norm = total_grad_norm / max(n_batches, 1)
    else:
        avg_grad_norm = None

    return avg_loss, avg_acc, avg_grad_norm


# ============================================================
# Main
# ============================================================

def main():
    print(f"Using device: {DEVICE}")
    print("Fashion-MNIST classes:", FASHION_CLASSES)
    print("Class mapping:", CLASS_MAP)

    train_loader, val_loader, test_loader = load_data()

    model = FashionQCNN4Class().to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("\nModel parameters:")
    print("theta_enc:", model.theta_enc.numel())
    print("theta_conv1:", model.theta_conv1.numel())
    print("theta_pool1:", model.theta_pool1.numel())
    print("theta_conv2:", model.theta_conv2.numel())
    print("theta_pool2:", model.theta_pool2.numel())
    print("total:", sum(p.numel() for p in model.parameters()))

    best_val_acc = 0.0
    best_state = None
    best_epoch = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, train_grad = run_epoch(
            model,
            train_loader,
            optimizer=optimizer,
        )

        val_loss, val_acc, _ = run_epoch(
            model,
            val_loader,
            optimizer=None,
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d} | "
            f"train loss={train_loss:.4f}, train acc={train_acc:.4f}, "
            f"grad norm={train_grad:.6e} | "
            f"val loss={val_loss:.4f}, val acc={val_acc:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc, _ = run_epoch(
        model,
        test_loader,
        optimizer=None,
    )

    print("\nBest validation accuracy:", best_val_acc)
    print("Best epoch:", best_epoch)
    print(f"Test loss={test_loss:.4f}, test acc={test_acc:.4f}")

    print("\nClass order:")
    print("0 -> T-shirt/top")
    print("1 -> Trouser")
    print("2 -> Sneaker")
    print("3 -> Bag")

    print("\nEncoding:")
    print("Pairwise fragment encoding with local 2x2 mean features")
    print("16x16 image -> four 8x8 patches")
    print("each 8x8 patch -> sixteen 2x2 local mean features")
    print("top-left and top-right share theta_enc[0]")
    print("bottom-left and bottom-right share theta_enc[1]")

    print("\nOutput probability mapping:")
    print("class 0 -> P(00)")
    print("class 1 -> P(01)")
    print("class 2 -> P(10)")
    print("class 3 -> P(11)")

    print("\nValidation confusion matrix:")
    print(confusion_matrix(model, val_loader))

    print("\nTest confusion matrix:")
    print(confusion_matrix(model, test_loader))


if __name__ == "__main__":
    main()

Using device: cpu
Fashion-MNIST classes: [0, 1, 7, 8]
Class mapping: {0: 0, 1: 1, 7: 2, 8: 3}

Model parameters:
theta_enc: 32
theta_conv1: 10
theta_pool1: 2
theta_conv2: 10
theta_pool2: 2
total: 56
Epoch 01 | train loss=1.3649, train acc=0.4094, grad norm=1.266403e+00 | val loss=1.1240, val acc=0.5381
Epoch 02 | train loss=1.0629, train acc=0.5589, grad norm=5.199352e-01 | val loss=0.9923, val acc=0.6050
Epoch 03 | train loss=0.9644, train acc=0.6228, grad norm=4.388069e-01 | val loss=0.9267, val acc=0.6619
Epoch 04 | train loss=0.9106, train acc=0.6553, grad norm=4.046000e-01 | val loss=0.8883, val acc=0.6681
Epoch 05 | train loss=0.8753, train acc=0.6646, grad norm=3.800302e-01 | val loss=0.8575, val acc=0.6787
Epoch 06 | train loss=0.8470, train acc=0.6690, grad norm=3.617622e-01 | val loss=0.8303, val acc=0.6888
Epoch 07 | train loss=0.8220, train acc=0.6778, grad norm=3.506183e-01 | val loss=0.8064, val acc=0.7050
Epoch 08 | train loss=0.8016, train acc=0.6848, grad norm=3.492974